# Week Exercise - week 4

Code Generation Exercise
- A code tool that automatically add docstring / comments.
- A code generation tool that automatically write unit test cases. 

Problem statement: To build a code tool that automatically add docstring / comments and generate unit test cases

In [ ]:
! uv pip install pytest==9.0.2

In [ ]:
! uv pip install mockito

In [12]:
import os
import io
import sys
import tempfile
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import subprocess
import pytest
from IPython.display import Markdown, display
from io import StringIO

In [ ]:
# Load environment variables in a file called .env
# Print the key prefixes to help with any debugging
# You can choose whichever providers you like - or all Ollama

load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
openrouter_base_url = os.getenv('OPENROUTER_BASE_URL')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:3]}")
else:
    print("OpenRouter API Key not set (and this is optional)")

In [14]:
# Connect to OpenAI, Anthropic and Google; comment out the Claude or Google lines if you're not using them

# openai = OpenAI()

client = OpenAI(api_key=openrouter_api_key, base_url=openrouter_base_url)

In [15]:
# constants
MODELS = [
    "openai/gpt-5-mini",
    "x-ai/grok-3-fast",
    "google/gemini-2.5-flash",
    "qwen/qwen3-coder:free",
    "meta-llama/llama-3.3-70b-versatile",
    "google/gemma-3-27b-it:free",
    "openai/gpt-4o-mini",
    "openai/gpt-5-nano",
    "openai/gpt-4.1-mini",
    "anthropic/claude-3-5-sonnet",
    "ollama/llama3.2"
]

In [16]:
# System prompts for code generation and docstrings comments and for unit tests

MAX_UNIT_TESTS = 5  # maximum allowed number of tests

SYSTEM_PROMPT_CODING_AGENT = """
You are a senior software engineer and system architect with deep expertise in Python and Spring Boot (Java).

Your goal is to produce correct, maintainable, and production-quality code.

LANGUAGE DETECTION
------------------
Determine the programming language from the user's request or code snippet.

If Python → produce Python code.
If Java → produce Spring Boot compatible Java code.

If unclear → ask for clarification instead of guessing.

IMPLEMENTATION RULES
--------------------
Follow professional software engineering practices:

1. Write clean, readable, idiomatic code.
2. Follow language best practices:
   - Python: PEP8 conventions
   - Java: standard Spring Boot conventions
3. Prefer simple and maintainable solutions over clever ones.
4. Avoid unnecessary complexity.
5. Validate inputs when appropriate.
6. Handle errors properly.

DOCUMENTATION
-------------
Add documentation for all public functions, classes, and methods.

Python:
Use PEP-257 docstrings including:
- Summary
- Args
- Returns
- Raises (when relevant)

Java:
Use Javadoc including:
- Description
- @param
- @return
- @throws

COMMENTS
--------
Add short inline comments only where logic may be difficult to understand.

Do NOT comment obvious code.

STRUCTURE
---------
If the request describes a component or service:

Python:
- Use functions or classes as appropriate.

Spring Boot:
- Use correct Spring annotations when applicable
  (@Service, @RestController, @Repository, etc).

If the user provides partial code:
- Complete the implementation.
- Improve documentation and clarity.

OUTPUT RULES
------------
Return only the source code. Output **complete, valid Python or Java** only.
Do not include explanations. Do not include markdown code fences.
Do not output partial or token-by-token lines; every line must be a complete statement or expression.
"""


SYSTEM_PROMPT_TEST_ENGINEER = f"""
You are a senior QA engineer specializing in automated testing.

Your task is to write unit tests for the provided code.

LANGUAGE DETECTION
------------------
Detect whether the code is written in Python or Java.

TEST COUNT RULE
---------------
Generate only the number of tests necessary to properly verify the behavior.

Guidelines:
- Write at least 1 test.
- Write at most {MAX_UNIT_TESTS} tests.
- Do NOT generate unnecessary or repetitive tests.

Focus on meaningful coverage rather than quantity.

PYTHON TESTING RULES
--------------------
Use pytest.

Assumptions:
- The code under test is in generated_code.py
- Import using:

    from generated_code import *

Test coverage should include when relevant:
- normal behavior
- edge cases
- invalid inputs
- expected exceptions

Use clear test names such as:

test_calculate_total_valid_input
test_calculate_total_zero
test_calculate_total_invalid_type


JAVA TESTING RULES
------------------
Use JUnit 5.

Assumptions:
- The class under test exists in the same package.
- Create a test class named <ClassName>Test.

Test coverage should include when relevant:
- normal behavior
- boundary cases
- invalid inputs
- expected exceptions

Use clear method names such as:

testCalculateTotalValidInput
testCalculateTotalZero
testCalculateTotalInvalidInput


OUTPUT RULES
------------
Return only the test code. Output **valid, executable** code only.
Do not include explanations. Do not include markdown fences.
Do not output partial or token-by-token lines; every line must be complete.
Include necessary imports (e.g. import pytest, from generated_code import ...). No duplicate imports.
Output complete test functions only. Do not write prose.
"""



In [17]:
def stream_completion(messages, model):
    """
    Generator that streams output from OpenRouter safely.
    Yields partial strings as the model streams.
    """

    stream = client.chat.completions.create(
        model=model,
        messages=messages,
        stream=True,
        max_tokens=8192
    )

    partial = ""

    for chunk in stream:
        # safety checks
        if not hasattr(chunk, "choices") or len(chunk.choices) == 0:
            continue  # skip empty chunks

        choice = chunk.choices[0]

        # Some early chunks may not have delta yet
        if not hasattr(choice, "delta") or choice.delta is None:
            continue

        content = getattr(choice.delta, "content", "")
        if content:
            partial += content
            yield partial


In [18]:
def generate_code_stream(prompt, model):

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT_CODING_AGENT},
        {"role": "user", "content": prompt}
    ]

    yield from stream_completion(messages, model)



def generate_tests_stream(code, model):
    """Stream test code (may truncate on long output)."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT_TEST_ENGINEER},
        {"role": "user", "content": code}
    ]
    yield from stream_completion(messages, model)


def _collapse_streaming_fragments(text):
    """If model output has one token/fragment per line (with -> with pytest -> ...), keep only full lines."""
    lines = text.split("\n")
    if len(lines) < 2:
        return text
    out = []
    i = 0
    while i < len(lines):
        line = lines[i]
        j = i + 1
        while j < len(lines):
            a, b = line.rstrip(), lines[j].rstrip()
            if a and b and b != a and b.startswith(a):
                line = lines[j]
                j += 1
            else:
                break
        out.append(line)
        i = j
    return "\n".join(out)

def generate_tests_full(code, model):
    """Single non-streaming request so the full test code is returned (avoids truncation)."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT_TEST_ENGINEER},
        {"role": "user", "content": code}
    ]
    resp = client.chat.completions.create(
        model=model,
        messages=messages,
        stream=False,
        max_tokens=8192
    )
    text = (resp.choices[0].message.content or "").strip()
    text = _collapse_streaming_fragments(text)
    return text


def generate_code_full(prompt, model):
    """Single non-streaming request so the full source code is returned (avoids fragment output)."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT_CODING_AGENT},
        {"role": "user", "content": prompt}
    ]
    resp = client.chat.completions.create(
        model=model,
        messages=messages,
        stream=False,
        max_tokens=8192
    )
    text = (resp.choices[0].message.content or "").strip()
    text = _collapse_streaming_fragments(text)
    return text


In [19]:
import os
import sys
import tempfile
from io import StringIO
import pytest
import re

def run_python_tests(code_str, test_str):
    """Safely run Python tests (pytest) from code and test strings. Returns a summary string."""
    t = (test_str or "").strip()
    if t.startswith("```"):
        first_nl = t.find("\n")
        t = t[first_nl + 1:] if first_nl != -1 else t[3:]
    if t.endswith("```"):
        t = t[:-3].rstrip()
    test_str = t
    try:
        test_str = _collapse_streaming_fragments(test_str)
    except NameError:
        pass
    import ast
    try:
        ast.parse(test_str)
    except SyntaxError as e:
        return f"Generated test code has a syntax error (line {e.lineno or '?'}): {e.msg}.\n\nThe generated tests may be truncated. Try **Generate Tests** again or fix the Unit Tests code manually."
    with tempfile.TemporaryDirectory() as tmpdir:
        code_file = os.path.join(tmpdir, "generated_code.py")
        test_file = os.path.join(tmpdir, "test_generated_code.py")
        with open(code_file, "w") as f:
            f.write((code_str or "").strip() if isinstance(code_str, str) else str(code_str or ""))
        with open(test_file, "w") as f:
            f.write(test_str)
        import subprocess
        proc = subprocess.run(
            [sys.executable, "-m", "pytest", "test_generated_code.py", "-q", "--tb=short"],
            capture_output=True, text=True, cwd=tmpdir, timeout=60
        )
        output = (proc.stdout or "") + (proc.stderr or "")
        m_passed = re.search(r"(\d+)\s+passed", output)
        m_failed = re.search(r"(\d+)\s+failed", output)
        passed = int(m_passed.group(1)) if m_passed else output.count("PASSED")
        failed = int(m_failed.group(1)) if m_failed else output.count("FAILED")
        total = passed + failed
        result = f"total tests: {total}\npassed: {passed}\nfailed: {failed}"
        if total == 0 and output.strip():
            result += "\n\n--- pytest output ---\n" + output.strip()
        return result

def run_code_agent(prompt, model):
    """Generate complete code in one request (no streaming fragments)."""
    full_code = generate_code_full(prompt, model)
    yield full_code


def run_test_agent(code, model):

    for chunk in generate_tests_stream(code, model):
        yield chunk


seen_imports = set()

def filter_imports(chunk: str) -> str:
    """
    Keep only unique import statements.
    """
    lines = chunk.split("\n")
    result = []
    for line in lines:
        stripped = line.strip()
        if stripped.startswith("import") or stripped.startswith("from"):
            if stripped not in seen_imports:
                seen_imports.add(stripped)
                result.append(line)
        else:
            result.append(line)
    return "\n".join(result)
    

def stream_tests_clean(code: str, model):
    """
    Stream Python tests from AI, clean duplicates and imports.
    """
    partial = ""
    emitted_lines = set()

    for chunk in run_test_agent(code, model):
        if not chunk:
            continue

        # Remove duplicate imports
        chunk = filter_imports(chunk)

        # Remove duplicate lines
        lines = chunk.split("\n")
        new_lines = []
        for line in lines:
            stripped = line.strip()
            if stripped and stripped not in emitted_lines:
                emitted_lines.add(stripped)
                new_lines.append(line)

        if new_lines:
            partial += "\n".join(new_lines) + "\n"
            yield partial


def run_tests(code, test_code, language="Python"):
    """
    Safely run Python tests or handle Java tests.
    Returns a summary string.
    """
    try:
        # Normalize: gr.Code can be str, None, or dict
        if code is None: code = ""
        elif isinstance(code, dict): code = code.get("code", code.get("text", "")) or ""
        else: code = str(code) if code else ""
        if test_code is None: test_code = ""
        elif isinstance(test_code, dict): test_code = test_code.get("code", test_code.get("text", "")) or ""
        else: test_code = str(test_code) if test_code else ""
        if not code.strip():
            return "No generated code. Generate code in **Generate Code** tab, then tests in **Generate Tests** tab, then Run Tests."
        if not test_code.strip():
            return "No generated tests. Generate tests in **Generate Tests** tab, then Run Tests."
        lang = (language or "Python").strip().lower() or "python"
        if lang == "python":
            return run_python_tests(code, test_code)
        else:
            lines = test_code.strip().split("\n")
            total_tests = sum(1 for l in lines if "void" in l.lower() and "test" in l.lower())
            return f"Java tests generated.\nEstimated total tests: {total_tests}\nCannot execute in Python environment."
    except Exception as e:
        return f"Run tests failed: {type(e).__name__}: {e}"


def generate_tests_wrapper(code_input_box, generated_code_box, model, language):
    global seen_imports
    seen_imports = set()  # reset imports

    code_to_use = generated_code_box.strip() or code_input_box.strip()
    if not code_to_use:
        return """Please provide the code first.
Do you want Python or Java (Spring Boot)?
Specify functionality and constraints:
- Primality test
- List/generate all primes up to N
- Web/API service
- CLI or library usage
"""

    if language.lower() == "python":
        # Use full (non-streaming) so we get complete test code and avoid truncation
        full_code = generate_tests_full(code_to_use, model)
        yield full_code
    else:
        # Java: consume generator and yield full response
        chunks = list(run_test_agent(code_to_use, model))
        yield "".join(chunks) if chunks else ""


In [ ]:
import gradio as gr

with gr.Blocks() as demo:

    gr.Markdown("# AI Coding Assistant")

    # -------------------------------
    # Model Selector
    # -------------------------------
    model_selector = gr.Dropdown(
        choices=MODELS,
        value=MODELS[0],
        label="Select Model"
    )

    # -------------------------------
    # Generate Code Tab
    # -------------------------------
    with gr.Tab("Generate Code"):

        prompt_input = gr.Textbox(
            label="Describe the code",
            lines=8
        )

        run_btn = gr.Button("Generate")

        code_output = gr.Code(
            label="Generated Code"
        )

        run_btn.click(
            run_code_agent,
            inputs=[prompt_input, model_selector],
            outputs=[code_output]
        )

    # -------------------------------
    # Generate Tests Tab
    # -------------------------------
    with gr.Tab("Generate Tests"):

        code_input = gr.Code(label="Paste Code")

        test_btn = gr.Button("Generate Tests")

        test_output = gr.Code(label="Unit Tests")

        # Optional: hidden language dropdown or auto-detected
        detected_language_input = gr.Dropdown(
            choices=["Python", "Java"],
            value="Python",  # default
            label="Programming Language"
        )

        test_btn.click(
            generate_tests_wrapper,
            inputs=[code_input, code_output, model_selector, detected_language_input],
            outputs=[test_output]
        )

    # -------------------------------
    # Run Tests Tab
    # -------------------------------
    with gr.Tab("Run Tests"):

        gr.Markdown("Uses the **Generated Code** from the Generate Code tab and **Unit Tests** from the Generate Tests tab.")
        test_results_output = gr.Textbox(label="Test Results", lines=5)
        run_tests_btn = gr.Button("Run Tests")

        detected_language_input_run = gr.Dropdown(
            choices=["Python", "Java"],
            value="Python",
            label="Programming Language"
        )

        # Automatically read from code_output and test_output (no copy-paste needed)
        run_tests_btn.click(
            run_tests,
            inputs=[code_output, test_output, detected_language_input_run],
            outputs=[test_results_output]
        )

# -------------------------------
# Launch on web
# -------------------------------
demo.launch(share=True)

